# Rule-based — Caro 11×11

Notebook thí nghiệm cho tác tử Rule-based. Thuật toán được import từ `agents/rule_based_agent.py`;
notebook chỉ dựng thế cờ, chạy tác tử và trình bày kết quả.

**Trạng thái hiện tại — Giai đoạn 1:** đã có luật thắng ngay, luật chặn thua và nước dự phòng theo vị trí.
Mục 1–4 có số liệu. Mục 5–6 (pattern, giải thích quyết định nâng cao) thuộc Giai đoạn 2; mục 7–10
(ablation, thi đấu thử, phân tích thất bại, tổng kết) thuộc Giai đoạn 3.

Chạy notebook bằng **Restart Kernel & Run All** từ kernel `.venv`.

In [1]:
from pathlib import Path
import os

# Chuyển về thư mục Caro để import module chung từ vị trí notebook.
if Path.cwd().name == 'notebooks' and (Path.cwd().parent / 'core').is_dir():
    os.chdir(Path.cwd().parent)
if not (Path.cwd() / 'core' / 'game_engine.py').is_file():
    raise RuntimeError('Hãy mở notebook từ thư mục caro-11x11 hoặc notebooks')

import platform
import sys
from collections import Counter

import numpy as np
import pandas as pd

from agents import RandomAgent
from agents.rule_based_agent import (DEFAULT_RADIUS, DIRECTIONS, RULE_BLOCK, RULE_FALLBACK, RULE_WIN,
                                     RuleBasedAgent, position_score, rank_key, winning_moves)
from config.game_config import BOARD_SIZE, WIN_LENGTH
from core import CaroGame
from core.game_utils import candidate_actions
from evaluation.metrics import MatchRecord, summarize_results

X, O = 1, -1
SIDE_NAME = {X: 'X', O: 'O'}


def left(df):
    # Căn trái nội dung và tiêu đề bảng để dễ đọc; chỉ dùng để hiển thị.
    return df.style.set_properties(**{'text-align': 'left'}).set_table_styles(
        [{'selector': 'th', 'props': [('text-align', 'left')]}])


def build_position(x_cells, o_cells, size=BOARD_SIZE, k=WIN_LENGTH):
    # Dựng thế cờ bằng engine: đi xen kẽ X, O theo hai danh sách (engine không có API nạp bàn cờ).
    assert len(x_cells) in (len(o_cells), len(o_cells) + 1), 'X đi trước nên số quân X = O hoặc O + 1'
    game = CaroGame(size, k)
    for i in range(len(x_cells) + len(o_cells)):
        game.step(x_cells[i // 2] if i % 2 == 0 else o_cells[i // 2])
    assert not game.is_terminal()
    return game

## 1. Câu hỏi nghiên cứu và phạm vi

**Câu hỏi nghiên cứu của cả notebook:** việc bổ sung nhận diện pattern có thực sự nâng cao chất lượng nước đi
so với việc chỉ dùng luật thắng ngay / chặn thua kết hợp điểm vị trí cơ bản hay không?

Giai đoạn 1 xây dựng **baseline** cho câu hỏi đó: phiên bản chỉ có luật thắng ngay, chặn thua và nước dự phòng.
Các giai đoạn sau so sánh phiên bản có pattern với chính baseline này.

**Phạm vi Giai đoạn 1:**
- Nhận diện nước tạo chuỗi từ 5 quân trở lên theo 4 hướng, đúng luật engine (freestyle: chuỗi dài hơn 5 vẫn thắng,
  bị chặn hai đầu vẫn thắng).
- Ưu tiên thắng ngay trước khi phòng thủ; tìm đe dọa của đối thủ trên **toàn bộ ô trống**.
- Khi chưa có đe dọa trực tiếp, chọn nước dự phòng hợp lệ theo điểm vị trí đơn giản.

**Giới hạn đã biết trước khi đo:**
- Luật chỉ nhìn **một nước**: thấy chuỗi 4 quân sắp thành 5, nhưng không thấy ba mở sắp thành bốn mở.
  Khi đối thủ đã có bốn mở (hai ô thắng), tác tử chỉ chặn được một ô.
- Nước dự phòng không đánh giá pattern (ba mở, bốn mở, chuỗi có khoảng trống); phần này thuộc Giai đoạn 2.

## 2. Môi trường và cấu hình

**Thứ tự luật** (dừng ở tầng đầu tiên có nước):

| Tầng | Tên trong mã | Khi nào kích hoạt | Tập ô được xét | Tập nước chấp nhận được |
| :--- | :--- | :--- | :--- | :--- |
| 1 | `thang_ngay` | Bên đang đi có ô tạo chuỗi ≥ 5 quân | Toàn bộ ô trống | Mọi ô thắng ngay |
| 2 | `chan_thua` | Đối thủ có ô thắng ngay ở lượt sau | Toàn bộ ô trống | Mọi ô thắng của đối thủ (đi vào là chặn) |
| 3 | `du_phong` | Không có hai trường hợp trên | Ứng viên bán kính 2 (`candidate_actions`) | Mọi ứng viên có điểm vị trí cao nhất |

**Điểm vị trí của tầng dự phòng:** `2 × (số quân mình) + 1 × (số quân đối thủ)` trong 8 ô kề.
Nối quân mình được ưu tiên hơn áp sát quân đối thủ. Đây là điểm baseline, không phải pattern.

**Phá hòa (dùng chung cho cả ba tầng):** điểm vị trí cao hơn → gần tâm hơn (khoảng cách Chebyshev, rồi Manhattan)
→ hàng nhỏ hơn → cột nhỏ hơn. Tác tử **hoàn toàn tất định**, không dùng số ngẫu nhiên nên không cần `seed`;
cùng một thế cờ luôn cho cùng một nước.

**Định nghĩa nước chấp nhận được khi có nhiều nước tương đương:** mọi nước trong cột cuối của bảng trên đều đúng
về mặt luật. Ví dụ đối thủ có bốn mở (hai ô thắng), chặn ô nào cũng đúng; kiểm thử chỉ yêu cầu nước nằm trong
tập này, không ép một ô cụ thể. Tập này được lưu trong `Decision.acceptable` để đối chiếu.

**Định nghĩa thống kê** (`agent.stats`, đặt lại mỗi lượt):
| Chỉ số | Cách đếm |
| :--- | :--- |
| `nodes_visited` | Số ô được đánh giá: số ô trống ở tầng 1, cộng số ô trống ở tầng 2 (nếu tới), cộng số ứng viên ở tầng 3 (nếu tới). |
| `simulations`, `pruned_nodes` | Luôn 0: tác tử không mô phỏng và không cắt tỉa. |
| `thinking_time` | Do `BaseAgent.choose_action` đo, đơn vị giây. Tác tử trả nước ngay, không dùng hết ngân sách 2,0 giây. |

In [2]:
agent = RuleBasedAgent()
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__)
config_table = pd.DataFrame([
    {'tham số': 'kích thước bàn', 'giá trị': BOARD_SIZE},
    {'tham số': 'độ dài thắng (win_length)', 'giá trị': agent.win_length},
    {'tham số': 'bán kính ứng viên tầng dự phòng', 'giá trị': agent.radius},
    {'tham số': 'hướng kiểm tra', 'giá trị': str(DIRECTIONS)},
    {'tham số': 'thứ tự luật', 'giá trị': ' → '.join([RULE_WIN, RULE_BLOCK, RULE_FALLBACK])},
    {'tham số': 'seed', 'giá trị': 'không dùng (tác tử tất định)'},
])
left(config_table)

Python 3.11.9 | numpy 2.4.6 | pandas 3.1.0


,tham số,giá trị
0,kích thước bàn,11
1,độ dài thắng (win_length),5
2,bán kính ứng viên tầng dự phòng,2
3,hướng kiểm tra,"((0, 1), (1, 0), (1, 1), (1, -1))"
4,thứ tự luật,thang_ngay → chan_thua → du_phong
5,seed,không dùng (tác tử tất định)


Ví dụ cách phá hòa ở tầng dự phòng: sau nước X ở tâm, O đến lượt. Mọi ô kề tâm có cùng điểm vị trí 1
(kề một quân đối thủ). **Kỳ vọng:** các ô kề thẳng (cách tâm Manhattan 1) đứng trước ô kề chéo; trong bốn ô kề
thẳng, ô có hàng nhỏ nhất là (4, 5) được chọn.

In [3]:
game = build_position([(5, 5)], [])
candidates = candidate_actions(game.board, DEFAULT_RADIUS)
ranked = sorted(candidates, key=lambda a: rank_key(game.board, a, O))[:6]
left(pd.DataFrame([{'thứ hạng': i + 1, 'ô': a, 'điểm vị trí': position_score(game.board, a, O),
                    'khóa phá hòa': rank_key(game.board, a, O)} for i, a in enumerate(ranked)]))

,thứ hạng,ô,điểm vị trí,khóa phá hòa
0,1,"(4, 5)",1,"(-1, 1, 1, 4, 5)"
1,2,"(5, 4)",1,"(-1, 1, 1, 5, 4)"
2,3,"(5, 6)",1,"(-1, 1, 1, 5, 6)"
3,4,"(6, 5)",1,"(-1, 1, 1, 6, 5)"
4,5,"(4, 4)",1,"(-1, 1, 2, 4, 4)"
5,6,"(4, 6)",1,"(-1, 1, 2, 4, 6)"


**Quan sát:** đúng như kỳ vọng, (4, 5) đứng đầu, sau đó là (5, 4), (5, 6), (6, 5) rồi mới tới ô chéo.
Bàn có 24 ứng viên (bán kính 2 quanh tâm); khóa phá hòa là bộ năm số, so sánh lần lượt từ trái sang.

## 3. Bộ vị trí chiến thuật

Mỗi thế cờ được dựng bằng engine qua chuỗi nước hợp lệ (`build_position`). Quân đệm đặt ở hàng 8 và 10, cách nhau
ít nhất 3 cột, nên không tạo thêm đe dọa. Bảng ghi bên đến lượt, tập nước chấp nhận được và lý do, đều xác định
**bằng tay trước khi chạy tác tử**. Thế "ba mở" để trống tập chấp nhận được: Giai đoạn 1 chỉ yêu cầu tác tử không
kích hoạt nhầm luật chặn; nước cụ thể sẽ được đánh giá ở Giai đoạn 2.

In [4]:
# Quân đệm: mọi đoạn 5 ô liên tiếp chứa tối đa 2 ô trong nhóm này.
FILLERS = [(10, 0), (10, 3), (10, 6), (10, 9), (8, 0), (8, 3), (8, 6), (8, 9)]

ALL_CELLS = [(r, c) for r in range(11) for c in range(11)]
X_PATTERN = [a for a in ALL_CELLS if (a[1] + 2 * a[0]) % 4 < 2]   # 61 ô, không có ba quân liền nhau
O_PATTERN = [a for a in ALL_CELLS if (a[1] + 2 * a[0]) % 4 >= 2]  # 60 ô

POSITIONS = [
    {'name': 'bàn trống', 'x': [], 'o': [],
     'acceptable': [(5, 5)], 'rule': RULE_FALLBACK,
     'reason': 'chưa có quân; quy ước candidate_actions trả về tâm'},
    {'name': 'X thắng ngang (bốn mở)', 'x': [(5, 3), (5, 4), (5, 5), (5, 6)], 'o': FILLERS[:4],
     'acceptable': [(5, 2), (5, 7)], 'rule': RULE_WIN,
     'reason': 'X có bốn quân liền, hai đầu trống: ô nào cũng thắng'},
    {'name': 'O chặn dọc (một đầu bị chặn)', 'x': [(3, 5), (4, 5), (5, 5), (6, 5)], 'o': [(2, 5)] + FILLERS[:2],
     'acceptable': [(7, 5)], 'rule': RULE_BLOCK,
     'reason': 'X còn đúng một ô thắng ở đầu dưới'},
    {'name': 'X chặn chéo chính từ góc', 'x': FILLERS[:4], 'o': [(0, 0), (1, 1), (2, 2), (3, 3)],
     'acceptable': [(4, 4)], 'rule': RULE_BLOCK,
     'reason': 'đầu trên đã là mép bàn, O chỉ thắng ở (4, 4)'},
    {'name': 'O thắng chéo phụ có khoảng trống', 'x': FILLERS[:5], 'o': [(2, 8), (3, 7), (5, 5), (6, 4)],
     'acceptable': [(4, 6)], 'rule': RULE_WIN,
     'reason': 'lấp khoảng trống giữa tạo năm quân'},
    {'name': 'X nối thành sáu quân', 'x': [(5, 1), (5, 2), (5, 3), (5, 5), (5, 6)], 'o': FILLERS[:5],
     'acceptable': [(5, 4)], 'rule': RULE_WIN,
     'reason': 'luật freestyle: chuỗi sáu quân vẫn thắng'},
    {'name': 'X thắng dù bị chặn hai đầu', 'x': [(5, 1), (5, 2), (5, 4), (5, 5)], 'o': [(5, 0), (5, 6)] + FILLERS[:2],
     'acceptable': [(5, 3)], 'rule': RULE_WIN,
     'reason': 'năm quân liền vẫn thắng dù hai đầu có quân O'},
    {'name': 'vừa có nước thắng vừa bị đe dọa', 'x': [(0, 0), (0, 1), (0, 2), (0, 3)],
     'o': [(5, 3), (5, 4), (5, 6), (5, 7)],
     'acceptable': [(0, 4)], 'rule': RULE_WIN,
     'reason': 'thắng ngay ở (0, 4) thay vì chặn (5, 5), dù (5, 5) gần tâm hơn'},
    {'name': 'đe dọa kép (O có bốn mở)', 'x': FILLERS[:4], 'o': [(5, 3), (5, 4), (5, 5), (5, 6)],
     'acceptable': [(5, 2), (5, 7)], 'rule': RULE_BLOCK,
     'reason': 'chặn được một đầu; đầu còn lại không chặn kịp (giới hạn của luật)'},
    {'name': 'ba mở chưa phải đe dọa', 'x': FILLERS[:3], 'o': [(5, 4), (5, 5), (5, 6)],
     'acceptable': None, 'rule': RULE_FALLBACK,
     'reason': 'O chưa có ô thắng ngay; không kích hoạt luật chặn'},
    {'name': 'chỉ còn một ô trống', 'x': X_PATTERN[:-1], 'o': O_PATTERN,
     'acceptable': [X_PATTERN[-1]], 'rule': RULE_FALLBACK,
     'reason': '120 quân, không có chuỗi năm; ô cuối là nước hợp lệ duy nhất'},
]

for pos in POSITIONS:
    pos['game'] = build_position(pos['x'], pos['o'])
left(pd.DataFrame([{'thế cờ': pos['name'], 'số quân': pos['game'].move_count,
                    'đến lượt': SIDE_NAME[pos['game'].current_player],
                    'luật kỳ vọng': pos['rule'],
                    'nước chấp nhận được': 'không áp đặt' if pos['acceptable'] is None else pos['acceptable'],
                    'lý do': pos['reason']} for pos in POSITIONS]))

,thế cờ,số quân,đến lượt,luật kỳ vọng,nước chấp nhận được,lý do
0,bàn trống,0,X,du_phong,"[(5, 5)]",chưa có quân; quy ước candidate_actions trả về tâm
1,X thắng ngang (bốn mở),8,X,thang_ngay,"[(5, 2), (5, 7)]","X có bốn quân liền, hai đầu trống: ô nào cũng thắng"
2,O chặn dọc (một đầu bị chặn),7,O,chan_thua,"[(7, 5)]",X còn đúng một ô thắng ở đầu dưới
3,X chặn chéo chính từ góc,8,X,chan_thua,"[(4, 4)]","đầu trên đã là mép bàn, O chỉ thắng ở (4, 4)"
4,O thắng chéo phụ có khoảng trống,9,O,thang_ngay,"[(4, 6)]",lấp khoảng trống giữa tạo năm quân
5,X nối thành sáu quân,10,X,thang_ngay,"[(5, 4)]",luật freestyle: chuỗi sáu quân vẫn thắng
6,X thắng dù bị chặn hai đầu,8,X,thang_ngay,"[(5, 3)]",năm quân liền vẫn thắng dù hai đầu có quân O
7,vừa có nước thắng vừa bị đe dọa,8,X,thang_ngay,"[(0, 4)]","thắng ngay ở (0, 4) thay vì chặn (5, 5), dù (5, 5) gần tâm hơn"
8,đe dọa kép (O có bốn mở),8,X,chan_thua,"[(5, 2), (5, 7)]",chặn được một đầu; đầu còn lại không chặn kịp (giới hạn của luật)
9,ba mở chưa phải đe dọa,6,X,du_phong,không áp đặt,O chưa có ô thắng ngay; không kích hoạt luật chặn


Bàn cờ của từng thế (dựng bằng engine, in bằng `CaroGame.render()`), kèm bên đến lượt và nước chấp nhận được.

In [5]:
for pos in POSITIONS:
    accepted = 'không áp đặt' if pos['acceptable'] is None else pos['acceptable']
    print(f"[{pos['name']}] — {SIDE_NAME[pos['game'].current_player]} đến lượt; chấp nhận: {accepted}")
    _ = pos['game'].render()
    print()

[bàn trống] — X đến lượt; chấp nhận: [(5, 5)]
    0  1  2  3  4  5  6  7  8  9 10
 0  .  .  .  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  .  .  .  .  .  .  .
 5  .  .  .  .  .  .  .  .  .  .  .
 6  .  .  .  .  .  .  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  .  .  .  .  .  .  .  .  .  .  .

[X thắng ngang (bốn mở)] — X đến lượt; chấp nhận: [(5, 2), (5, 7)]
    0  1  2  3  4  5  6  7  8  9 10
 0  .  .  .  .  .  .  .  .  .  .  .
 1  .  .  .  .  .  .  .  .  .  .  .
 2  .  .  .  .  .  .  .  .  .  .  .
 3  .  .  .  .  .  .  .  .  .  .  .
 4  .  .  .  .  .  .  .  .  .  .  .
 5  .  .  .  X  X  X  X  .  .  .  .
 6  .  .  .  .  .  .  .  .  .  .  .
 7  .  .  .  .  .  .  .  .  .  .  .
 8  .  .  .  .  .  .  .  .  .  .  .
 9  .  .  .  .  .  .  .  .  .  .  .
10  O  .  .  O  .  .  O  .  .  O  .

[O chặn dọc (một đầu 

## 4. Kiểm chứng luật thắng và chặn

### 4.1. Bộ vị trí chiến thuật

**Kỳ vọng trước khi chạy:** ở mọi thế, luật kích hoạt đúng cột "luật kỳ vọng" và nước chọn nằm trong tập chấp
nhận được. Sau nước của tác tử, engine xác nhận: thế "thắng" kết thúc với bên đến lượt thắng; thế "chặn" có một
đe dọa thì đối thủ không còn ô thắng ngay. Kiểm tra sau cùng dùng `clone()` + `step()` của engine, không dùng
hàm của tác tử.

In [6]:
def engine_winning_cells(game):
    # Các ô giúp bên đến lượt thắng ngay, kiểm tra bằng engine (clone rồi step).
    outcome = []
    for a in game.legal_actions():
        g = game.clone()
        g.step(a)
        if g.result() == game.current_player:
            outcome.append(a)
    return outcome


rows = []
for pos in POSITIONS:
    game = pos['game'].clone()
    player = game.current_player
    action = agent.choose_action(game.board, player)
    decision = agent.explain_last_action()
    in_acceptable = pos['acceptable'] is None or action in pos['acceptable']
    game.step(action)
    if game.is_terminal():
        after = {1: 'X thắng', -1: 'O thắng', 0: 'hòa'}[game.result()]
    else:
        after = f"{SIDE_NAME[game.current_player]} còn {len(engine_winning_cells(game))} ô thắng ngay"
    rows.append({'thế cờ': pos['name'], 'luật': decision.rule, 'đúng luật?': decision.rule == pos['rule'],
                 'nước chọn': action, 'trong tập chấp nhận?': in_acceptable,
                 'tập của tác tử': list(decision.acceptable) if len(decision.acceptable) <= 3 else f'{len(decision.acceptable)} ô',
                 'sau nước đi (engine)': after, 'ô đã xét': agent.stats['nodes_visited']})
position_table = pd.DataFrame(rows)
print(f"Đúng luật: {position_table['đúng luật?'].sum()}/{len(position_table)}; "
      f"nước trong tập chấp nhận: {position_table['trong tập chấp nhận?'].sum()}/{len(position_table)}")
left(position_table)

Đúng luật: 11/11; nước trong tập chấp nhận: 11/11


,thế cờ,luật,đúng luật?,nước chọn,trong tập chấp nhận?,tập của tác tử,sau nước đi (engine),ô đã xét
0,bàn trống,du_phong,True,"(5, 5)",True,"[(5, 5)]",O còn 0 ô thắng ngay,243
1,X thắng ngang (bốn mở),thang_ngay,True,"(5, 7)",True,"[(5, 2), (5, 7)]",X thắng,113
2,O chặn dọc (một đầu bị chặn),chan_thua,True,"(7, 5)",True,"[(7, 5)]",X còn 0 ô thắng ngay,228
3,X chặn chéo chính từ góc,chan_thua,True,"(4, 4)",True,"[(4, 4)]",O còn 0 ô thắng ngay,226
4,O thắng chéo phụ có khoảng trống,thang_ngay,True,"(4, 6)",True,"[(4, 6)]",O thắng,112
5,X nối thành sáu quân,thang_ngay,True,"(5, 4)",True,"[(5, 4)]",X thắng,111
6,X thắng dù bị chặn hai đầu,thang_ngay,True,"(5, 3)",True,"[(5, 3)]",X thắng,113
7,vừa có nước thắng vừa bị đe dọa,thang_ngay,True,"(0, 4)",True,"[(0, 4)]",X thắng,113
8,đe dọa kép (O có bốn mở),chan_thua,True,"(5, 7)",True,"[(5, 2), (5, 7)]",O còn 1 ô thắng ngay,226
9,ba mở chưa phải đe dọa,du_phong,True,"(4, 5)",True,"[(4, 5), (6, 5)]",O còn 0 ô thắng ngay,286


**Quan sát so với kỳ vọng:** cả 11 thế đều kích hoạt đúng luật và chọn nước trong tập chấp nhận được.
- Các thế "thắng" đều kết thúc bằng chiến thắng của bên đến lượt, kể cả khi nối thành sáu quân hoặc bị chặn hai đầu.
- Thế **vừa có nước thắng vừa bị đe dọa**: tác tử đi (0, 4) để thắng chứ không chặn (5, 5), dù (5, 5) xếp hạng vị trí
  tốt hơn. Điều này cho thấy thứ tự luật quyết định, không phải điểm vị trí.
- Thế **đe dọa kép**: tác tử chặn (5, 7) (cùng điểm vị trí với (5, 2) nhưng gần tâm hơn), và O vẫn còn 1 ô thắng
  ngay. Đây là giới hạn tự nhiên của luật nhìn một nước, không phải lỗi; muốn tránh phải chặn từ lúc O mới có ba mở.
- Thế **ba mở** không kích hoạt luật chặn, đúng phạm vi Giai đoạn 1. Tác tử đi (4, 5), sát quân O nhưng không chặn
  đầu chuỗi (5, 3) hay (5, 7), nên ở lượt sau O có thể tạo bốn mở. Đây chính là chỗ pattern ở Giai đoạn 2 cần cải thiện.
- Cột "ô đã xét" khớp định nghĩa ở mục 2: tầng thắng xét đúng số ô trống, tầng chặn gấp đôi, bàn trống là
  121 + 121 + 1 = 243.

### 4.2. Quét bốn hướng, hai bên, giữa bàn và sát biên

Mục 4.1 chỉ có một số hướng. Ô dưới đây sinh tự động chuỗi bốn quân cho **4 hướng × 2 bên × 3 vị trí**
(giữa bàn, sát biên, có khoảng trống ở giữa) rồi kiểm tra cả hai vai trò: bên có chuỗi đến lượt (phải thắng)
và bên kia đến lượt (phải chặn).

**Ô thắng kỳ vọng được tính tay theo hình học:** với chuỗi liền bốn quân là hai ô ở hai đầu nếu còn trong bàn;
với chuỗi có khoảng trống là chính ô trống đó.

In [7]:
def four_stones(start, direction, with_gap):
    # Bốn quân theo hướng; with_gap=True bỏ trống vị trí thứ 3 (mẫu XX_XX, 5 ô).
    dr, dc = direction
    steps = [0, 1, 3, 4] if with_gap else [0, 1, 2, 3]
    stones = [(start[0] + k * dr, start[1] + k * dc) for k in steps]
    if with_gap:
        expected = [(start[0] + 2 * dr, start[1] + 2 * dc)]
    else:
        ends = [(start[0] - dr, start[1] - dc), (start[0] + 4 * dr, start[1] + 4 * dc)]
        expected = [a for a in ends if 0 <= a[0] < 11 and 0 <= a[1] < 11]
    return stones, sorted(expected)


DIRECTION_NAME = {(0, 1): 'ngang', (1, 0): 'dọc', (1, 1): 'chéo chính', (1, -1): 'chéo phụ'}
# Điểm bắt đầu cho từng hướng: giữa bàn / sát biên (chuỗi chạm mép) / có khoảng trống.
START = {
    (0, 1): {'giữa': (5, 3), 'sát biên': (0, 0), 'khoảng trống': (4, 2)},
    (1, 0): {'giữa': (3, 5), 'sát biên': (0, 10), 'khoảng trống': (2, 4)},
    (1, 1): {'giữa': (3, 3), 'sát biên': (0, 0), 'khoảng trống': (2, 2)},
    (1, -1): {'giữa': (3, 7), 'sát biên': (0, 10), 'khoảng trống': (2, 8)},
}

rows = []
for direction, starts in START.items():
    for kind, start in starts.items():
        stones, expected = four_stones(start, direction, kind == 'khoảng trống')
        for owner in (X, O):
            for role in ('thắng', 'chặn'):
                to_move = owner if role == 'thắng' else -owner
                # Bên đến lượt là X thì hai bên bằng quân, là O thì X nhiều hơn một quân.
                n_fillers = 4 if to_move == X else (3 if owner == X else 5)
                fillers = [a for a in FILLERS if a not in stones][:n_fillers]
                x_cells, o_cells = (stones, fillers) if owner == X else (fillers, stones)
                game = build_position(x_cells, o_cells)
                assert game.current_player == to_move
                action = agent.choose_action(game.board, to_move)
                decision = agent.explain_last_action()
                rows.append({'hướng': DIRECTION_NAME[direction], 'vị trí': kind, 'bên có chuỗi': SIDE_NAME[owner], 'vai': role,
                             'ô thắng kỳ vọng': expected, 'nước chọn': action,
                             'đạt': action in expected and decision.rule == (RULE_WIN if role == 'thắng' else RULE_BLOCK)})
sweep = pd.DataFrame(rows)
print(f'Tổng số ca: {len(sweep)}; đạt: {sweep["đạt"].sum()}')
left(sweep.pivot_table(index=['hướng', 'vị trí'], columns=['bên có chuỗi', 'vai'], values='đạt',
                      aggfunc=lambda s: '✓' if s.all() else '✗'))

Tổng số ca: 48; đạt: 48


**Quan sát so với kỳ vọng:** 48/48 ca đạt: ở cả 4 hướng, cả X và O, giữa bàn, sát biên và có khoảng trống, tác tử
đều thắng khi đến lượt và chặn đúng ô khi đối thủ có chuỗi. Các ca sát biên chỉ có một ô thắng vì đầu kia là
mép bàn, và tác tử chọn đúng ô còn lại. Chi tiết từng ca nằm trong biến `sweep`; bộ unit test
`tests/test_rule_based_agent.py` kiểm tra lại cùng các tình huống này kèm thế bị chặn hai đầu và chuỗi sáu quân.

### 4.3. Ván hoàn chỉnh với `RandomAgent`

Mục tiêu là xác nhận tác tử **chơi trọn ván** với nước hợp lệ ở mọi lượt, và áp dụng đúng luật trong ván thật:
ở mỗi lượt của Rule-based, engine liệt kê các ô thắng ngay của nó (để kiểm tra "bỏ lỡ nước thắng"), và sau nước
của nó liệt kê các ô thắng ngay của đối thủ (để đếm "để lọt đe dọa").

Cấu hình: 10 ván, Random dùng `seed` 0–4, mỗi seed chơi hai ván đổi bên đi trước/sau. Đây chỉ là kiểm tra tích hợp,
**không phải thi đấu chính thức** (mục 8 thuộc Giai đoạn 3); W/D/L tổng hợp bằng `evaluation/metrics.py`.

In [8]:
records, details = [], []
for seed in range(5):
    for rule_side in (X, O):
        game = CaroGame()
        rule, random_agent = RuleBasedAgent(), RandomAgent(seed=seed)
        rule_counts, missed_wins, missed_blocks = Counter(), 0, 0
        while not game.is_terminal():
            player = game.current_player
            if player == rule_side:
                own_wins = engine_winning_cells(game)
                action = rule.choose_action(game.board, player)
                rule_counts[rule.explain_last_action().rule] += 1
                game.step(action)
                if own_wins and game.result() != player:
                    missed_wins += 1
                if not game.is_terminal() and engine_winning_cells(game):
                    missed_blocks += 1
            else:
                game.step(random_agent.choose_action(game.board, player))
        name_x, name_o = ('Rule-based', 'Ngẫu nhiên') if rule_side == X else ('Ngẫu nhiên', 'Rule-based')
        records.append(MatchRecord(name_x, name_o, game.result()))
        outcome = game.result() * rule_side
        details.append({'seed Random': seed, 'Rule-based cầm': SIDE_NAME[rule_side], 'số nước': game.move_count,
                         'kết quả (Rule-based)': {1: 'thắng', 0: 'hòa', -1: 'thua'}[outcome],
                         RULE_WIN: rule_counts[RULE_WIN], RULE_BLOCK: rule_counts[RULE_BLOCK],
                         RULE_FALLBACK: rule_counts[RULE_FALLBACK],
                         'bỏ lỡ nước thắng': missed_wins, 'để lọt đe dọa': missed_blocks})
summary = summarize_results(records, 'Rule-based')
print(f"Rule-based: {summary['wins']} thắng / {summary['draws']} hòa / {summary['losses']} thua "
      f"trên {summary['total']} ván, điểm {summary['score']:.2f}")
left(pd.DataFrame(details))

Rule-based: 10 thắng / 0 hòa / 0 thua trên 10 ván, điểm 1.00


,seed Random,Rule-based cầm,số nước,kết quả (Rule-based),thang_ngay,chan_thua,du_phong,bỏ lỡ nước thắng,để lọt đe dọa
0,0,X,21,thắng,1,0,10,0,0
1,0,O,18,thắng,1,0,8,0,0
2,1,X,21,thắng,1,0,10,0,0
3,1,O,18,thắng,1,0,8,0,0
4,2,X,19,thắng,1,0,9,0,0
5,2,O,18,thắng,1,0,8,0,0
6,3,X,19,thắng,1,0,9,0,0
7,3,O,18,thắng,1,0,8,0,0
8,4,X,21,thắng,1,0,10,0,0
9,4,O,18,thắng,1,0,8,0,0


**Quan sát:** cả 10 ván đều chơi trọn với nước hợp lệ, mỗi ván 18–21 nước; Rule-based thắng 10/10. Cột "bỏ lỡ nước thắng" và
"để lọt đe dọa" đều bằng 0: mỗi khi có ô thắng ngay, tác tử đều kết thúc ván ngay lượt đó, và sau mỗi nước của nó
Random không còn ô thắng ngay nào. Luật chặn không kích hoạt lần nào vì Random không tự tạo được chuỗi bốn quân
trong 10 ván này; mỗi ván, Rule-based dùng luật thắng đúng một lần ở nước cuối, các nước còn lại là dự phòng.

**Giới hạn:** Random là đối thủ rất yếu, nên kết quả này chỉ xác nhận tích hợp và tính đúng của luật, không đo
sức mạnh. Luật chặn đã được kiểm chứng ở mục 4.1–4.2 và trong unit test; chất lượng nước dự phòng sẽ được
cải thiện và đo lại sau khi có pattern (Giai đoạn 2–3).

## 5–10. Các mục chưa thực hiện

- **Mục 5–6** (định nghĩa pattern, độ ưu tiên, giải thích quyết định khi xung đột luật): Giai đoạn 2.
- **Mục 7–10** (ablation bật/tắt pattern, thi đấu thử với `RandomAgent`, phân tích thất bại, tổng kết và hướng dẫn
  tái lập): Giai đoạn 3.

Các mục này chưa có số liệu.